# Pipeline d'Extraction des Données Strava

Ce notebook détaille les différentes étapes de l'extraction des données depuis l'API Strava jusqu'à leur sauvegarde sur le stockage cloud Onyxia (MinIO).

## Étape 1 : Initialisation et Connexion

Dans cette première étape, nous chargeons nos clés d'authentification depuis le fichier `.env` et nous définissons les chemins vers nos données.

In [ ]:
import os
import time
import requests
import pandas as pd
import s3fs
from pathlib import Path
from dotenv import load_dotenv

# 1. Définition du chemin racine (depuis le dossier notebooks)
# Si tu lances ce notebook depuis /notebooks, la racine est le dossier parent.
ROOT_DIR = Path().resolve().parent

# 2. Chargement du fichier .env
load_dotenv(dotenv_path=ROOT_DIR / ".env")
print("✅ Clés d'environnement chargées.")

# 3. Vérification rapide des variables essentielles
print(f"S3 Endpoint: {os.getenv('AWS_S3_ENDPOINT')}")
# Ne pas afficher les clés complètes par sécurité
print(f"Strava Client ID: {os.getenv('STRAVA_CLIENT_ID')}")

## Étape 2 : L'Authentification Strava

Pour interagir avec l'API Strava, nous avons besoin d'un "Access Token" (jeton d'accès) temporaire. Nous l'obtenons en échangeant notre "Refresh Token" (jeton de rafraîchissement) qui a une durée de vie plus longue.

**Important :** Si tu n'as pas de Refresh Token valide, tu devras d'abord exécuter le script `auth.py` pour autoriser l'application.

In [ ]:
def get_new_access_token():
    """Échange le refresh_token contre un access_token temporaire"""
    print("🔄 Rafraîchissement du jeton Strava...")
    payload = {
        'client_id': os.getenv('STRAVA_CLIENT_ID'),
        'client_secret': os.getenv('STRAVA_CLIENT_SECRET'),
        'refresh_token': os.getenv('STRAVA_REFRESH_TOKEN'),
        'grant_type': "refresh_token"
    }
    res = requests.post("https://www.strava.com/oauth/token", data=payload)
    res.raise_for_status()
    
    token = res.json()['access_token']
    print("✅ Jeton d'accès récupéré avec succès !")
    return token

# Exécution de l'authentification
try:
    access_token = get_new_access_token()
except Exception as e:
    print(f"❌ Erreur lors de l'obtention du token : {e}")

## Étape 3 : L'Extraction des Activités

Maintenant que nous sommes authentifiés, nous pouvons interroger l'API pour récupérer l'historique complet des activités. L'API renvoie les données par "pages" (pagination). Nous devons donc faire plusieurs requêtes pour tout récupérer.

In [ ]:
def fetch_all_activities(token):
    """Récupère l'intégralité des activités avec pagination"""
    print("📡 Récupération de l'historique complet Strava...")
    url = "https://www.strava.com/api/v3/athlete/activities"
    headers = {'Authorization': f'Bearer {token}'}
    
    all_activities = []
    page = 1
    per_page = 200 # Maximum autorisé par Strava

    while True:
        print(f"   📥 Lecture de la page {page}...")
        params = {'per_page': per_page, 'page': page}
        res = requests.get(url, headers=headers, params=params)
        res.raise_for_status()
        
        data = res.json()
        
        # Si la page est vide, on a fini !
        if not data:
            break
            
        all_activities.extend(data)
        page += 1
        
        # Petite pause pour respecter les limites de l'API (Rate Limiting)
        time.sleep(0.5)

    print(f"✅ Terminé ! {len(all_activities)} activités récupérées au total.")
    return pd.DataFrame(all_activities)

# Exécution de l'extraction
try:
    df_activites = fetch_all_activities(access_token)
    
    # Aperçu des données récupérées
    print("\nAperçu des premières lignes :")
    display(df_activites.head(3))
except Exception as e:
    print(f"❌ Erreur lors de l'extraction : {e}")

## Étape 4 : L'Export vers le Cloud (Onyxia)

Une fois les données récupérées sous forme de DataFrame Pandas, nous allons les sauvegarder en format CSV directement sur notre espace de stockage objet (S3/MinIO) sur Onyxia. C'est la "Zone Bronze" de notre Data Lake.

In [ ]:
def upload_to_onyxia(df):
    """Envoie le DataFrame en CSV sur MinIO (Onyxia)"""
    print("☁️ Envoi vers Onyxia (Zone Bronze)...")
    
    # Configuration de la connexion S3
    fs = s3fs.S3FileSystem(
        client_kwargs={'endpoint_url': f"https://{os.getenv('AWS_S3_ENDPOINT')}"},
        key=os.getenv('AWS_ACCESS_KEY_ID'),
        secret=os.getenv('AWS_SECRET_ACCESS_KEY'),
        token=os.getenv('AWS_SESSION_TOKEN')
    )

    target_path = "paleo/donnees_strava/activites_brutes.csv"

    # Écriture du fichier
    with fs.open(target_path, 'w', encoding='utf-8') as f:
        df.to_csv(f, index=False)
        
    print(f"🚀 Succès ! Fichier envoyé vers : {target_path}")

# Exécution de l'envoi
if not df_activites.empty:
    try:
        upload_to_onyxia(df_activites)
    except Exception as e:
        print(f"❌ Erreur lors de l'envoi sur Onyxia : {e}")
else:
    print("⚠️ Le DataFrame est vide, annulation de l'envoi.")

## Étape 5 : Test de l'Import depuis le Cloud

Pour s'assurer que notre sauvegarde a bien fonctionné, nous allons télécharger le fichier fraîchement créé depuis Onyxia et l'enregistrer dans notre dossier local `data/raw/`.

In [ ]:
def download_from_onyxia():
    """Télécharge le fichier brut depuis Onyxia vers data/raw/"""
    print("☁️ Connexion au stockage Onyxia pour le téléchargement...")
    
    # Cible le dossier data/raw à la racine
    DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
    DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

    chemin_s3 = "paleo/donnees_strava/activites_brutes.csv"
    chemin_local = DATA_RAW_DIR / "activites_brutes_test.csv"
    
    try:
        fs = s3fs.S3FileSystem(
            client_kwargs={'endpoint_url': f"https://{os.getenv('AWS_S3_ENDPOINT')}"},
            key=os.getenv('AWS_ACCESS_KEY_ID'),
            secret=os.getenv('AWS_SECRET_ACCESS_KEY'),
            token=os.getenv('AWS_SESSION_TOKEN')
        )
        
        print(f"📥 Téléchargement depuis : {chemin_s3}")
        fs.get(chemin_s3, str(chemin_local))
        print(f"✅ Succès ! Fichier test téléchargé dans : {chemin_local}")
        
    except Exception as e:
        print(f"❌ Erreur lors du téléchargement : {e}")

# Exécution du test de téléchargement
download_from_onyxia()